In [3]:
!pip install rouge-score bert-score torch

  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
  Using cached torch-2.12.1-cp314-cp314-win_amd64.whl.metadata (31 kB)
  Using cached transformers-5.12.1-py3-none-any.whl.metadata (33 kB)
  Using cached requests-2.34.2-py3-none-any.whl.metadata (4.8 kB)
  Using cached filelock-3.29.4-py3-none-any.whl.metadata (2.0 kB)
  Using cached setuptools-81.0.0-py3-none-any.whl.metadata (6.6 kB)
  Using cached sympy-1.14.0-py3-none-any.whl.metadata (12 kB)
  Using cached networkx-3.6.1-py3-none-any.whl.metadata (6.8 kB)
  Using cached jinja2-3.1.6-py3-none-any.whl.metadata (2.9 kB)
  Using cached fsspec-2026.6.0-py3-none-any.whl.metadata (10 kB)
  Using cached mpmath-1.3.0-py3-none-any.whl.metadata (8.6 kB)
  Using cach

In [1]:
!pip install ollama psutil nltk

   ---------------------------------------- 0.0/1.6 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.6 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.6 MB ? eta -:--:--
   ------ --------------------------------- 0.3/1.6 MB ? eta -:--:--
   ------ --------------------------------- 0.3/1.6 MB ? eta -:--:--
   ------------- -------------------------- 0.5/1.6 MB 599.9 kB/s eta 0:00:02
   ------------- -------------------------- 0.5/1.6 MB 599.9 kB/s eta 0:00:02
   -------------------- ------------------- 0.8/1.6 MB 657.8 kB/s eta 0:00:02
   -------------------- ------------------- 0.8/1.6 MB 657.8 kB/s eta 0:00:02
   --------------------------- ------------ 1.0/1.6 MB 699.0 kB/s eta 0:00:01
   --------------------------------- ------ 1.3/1.6 MB 706.6 kB/s eta 0:00:01
   --------------------------------- ------ 1.3/1.6 MB 706.6 kB/s eta 0:00:01
   ---------------------------------------- 1.6/1.6 MB 711.9 kB/s  0:00:02
   -------------------------------

In [1]:
import nltk
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\User\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [1]:
from rouge_score import rouge_scorer
from bert_score import score as bertscore

c:\Users\BLACKBOX\.anaconda-desktop\micromamba\envs\cuda\envs\condaenv1\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
from rouge_score import rouge_scorer
from bert_score import score as bertscore_calc

def evaluate_summary(generated_summary, reference_summary):
    # Guard rail against empty generations or errors
    if not generated_summary or not reference_summary or "Error during inference:" in str(generated_summary):
        return {
            "ROUGE1": 0.0,
            "ROUGE2": 0.0,
            "ROUGEL": 0.0,
            "BERTScore": 0.0
        }

    scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)
    scores = scorer.score(str(reference_summary), str(generated_summary))

    try:
        # Utilizing a fast lightweight backbone to calculate semantic embeddings quickly
        P, R, F1 = bertscore_calc(
            [str(generated_summary)], 
            [str(reference_summary)], 
            lang="en", 
            model_type="microsoft/deberta-v3-small", 
            verbose=False
        )
        bert_f1 = F1.mean().item()
    except Exception:
        bert_f1 = 0.0

    return {
        "ROUGE1": scores['rouge1'].fmeasure,
        "ROUGE2": scores['rouge2'].fmeasure,
        "ROUGEL": scores['rougeL'].fmeasure,
        "BERTScore": bert_f1
    }

In [14]:
import time
import os
import psutil
import pandas as pd
import ollama
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

# Ensure necessary NLP resources are downloaded
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('stopwords', quiet=True)

# 1. LOAD YOUR KAGGLE DATASET
csv_path = 'datasets/scisumm.csv' 

if not os.path.exists(csv_path):
    raise FileNotFoundError(f"Please make sure the dataset file is named '{csv_path}' and placed in the correct path.")

df = pd.read_csv(csv_path)
print("Dataset columns:", df.columns.tolist())

# Dynamically map text and summary columns based on standard naming variations
text_column = 'text' if 'text' in df.columns else ('document' if 'document' in df.columns else df.columns[0])

if 'summary' in df.columns:
    summary_column = 'summary'
elif 'abstract' in df.columns:
    summary_column = 'abstract'
else:
    summary_column = df.columns[1] # Fallback to the second column

print(f"Mapping Input Text to: '{text_column}' | Mapping Ground Truth Summary to: '{summary_column}'")

# Process the first 10 papers for testing
sample_df = df.head(1000)

# 2. PROMPT COMPRESSION FUNCTION
def compress_prompt(text):
    if not isinstance(text, str):
        return ""
    stop_words = set(stopwords.words('english'))
    word_tokens = word_tokenize(text)
    compressed_tokens = [w for w in word_tokens if not w.lower() in stop_words]
    return " ".join(compressed_tokens)

# 3. EXPERIMENTAL TESTING ENGINE
results = []

def run_test(pipeline_name, model_name, input_text, reference_summary, paper_id):
    truncated_text = " ".join(str(input_text).split()[:800])
    
    process = psutil.Process()
    start_mem = process.memory_info().rss / (1024 * 1024) # MB
    start_time = time.time()
    
    try:
        response = ollama.chat(model=model_name, messages=[
            {'role': 'user', 'content': f"Summarize this scientific text in two sentences: {truncated_text}"}
        ])
        output_text = response['message']['content']
    except Exception as e:
        output_text = f"Error during inference: {str(e)}"
        
    end_time = time.time()
    end_mem = process.memory_info().rss / (1024 * 1024) # MB
    
    latency = end_time - start_time
    memory_used = max(0, end_mem - start_mem)

    metrics = evaluate_summary(output_text, reference_summary)
    
    return {
        'Paper_ID': paper_id,
        'Pipeline': pipeline_name,
        'Latency_Sec': round(latency, 3),
        'RAM_Used_MB': round(memory_used, 2),
        'Output_Word_Count': len(output_text.split()),
        "Compression_Ratio": round(len(output_text.split()) / max(1, len(truncated_text.split())), 4),
        "ROUGE1": round(metrics["ROUGE1"], 4),
        "ROUGE2": round(metrics["ROUGE2"], 4),
        "ROUGEL": round(metrics["ROUGEL"], 4),
        "BERTScore": round(metrics["BERTScore"], 4),
        'Summary': output_text 
    }

# 4. RUN THE COMPARATIVE EXPERIMENT LOOP
print("\nStarting optimization benchmarks...")

for idx, row in sample_df.iterrows():
    raw_text = row[text_column]
    ref_summary = row[summary_column]
    print(f"Processing Paper {idx + 1}/1000...", end="\r")
    
    # Baseline
    res_base = run_test("Baseline", "phi3", raw_text, ref_summary, idx)
    
    # Pipeline A: Quantized Model
    res_pipe_a = run_test("Quantized_Model", "phi3:3.8b-mini-4k-instruct-q4_K_M", raw_text, ref_summary, idx)
    
    # Pipeline B: Prompt Compression
    compressed_text = compress_prompt(raw_text)
    res_pipe_b = run_test("Prompt_Compression", "phi3", compressed_text, ref_summary, idx)

    # Pipeline C: Quantized Model + Prompt Compression (Hybrid)
    res_pipe_c = run_test("Quantized_Prompt_Compression", "phi3:3.8b-mini-4k-instruct-q4_K_M", compressed_text, ref_summary, idx)
    
    results.extend([res_base, res_pipe_a, res_pipe_b, res_pipe_c])

print("\nAll experiments complete successfully!")

Dataset columns: ['text', 'summary']
Mapping Input Text to: 'text' | Mapping Ground Truth Summary to: 'summary'

Starting optimization benchmarks...


HTTP Error 504 thrown while requesting HEAD https://huggingface.co/microsoft/deberta-v3-small/resolve/main/tokenizer_config.json
Retrying in 1s [Retry 1/5].


HTTP Error 504 thrown while requesting HEAD https://huggingface.co/microsoft/deberta-v3-small/resolve/main/tokenizer_config.json
Retrying in 1s [Retry 1/5].


Processing Paper 1000/1000...
All experiments complete successfully!


In [15]:
# 5. CONVERT THE LOGGED METRICS INTO A DATAFRAME
results_df = pd.DataFrame(results)

# Only pivot the columns that we are absolutely certain exist in results_df
available_metrics = ['Latency_Sec', 'RAM_Used_MB', 'Output_Word_Count', 'ROUGE1', 'ROUGE2', 'ROUGEL', 'BERTScore']

pivot_df = results_df.pivot(
    index='Paper_ID',
    columns='Pipeline',
    values=available_metrics
)

# Swap structural levels so Pipeline is on top
pivot_df = pivot_df.swaplevel(0, 1, axis=1)

pipeline_order = [
    "Baseline",
    "Quantized_Model",
    "Prompt_Compression",
    "Quantized_Prompt_Compression"
]

# Reindex layout structure cleanly
pivot_df = pivot_df.reindex(
    columns=pd.MultiIndex.from_product([pipeline_order, available_metrics])
)

# Calculate averages for performance columns
average_row = pivot_df.mean()
pivot_df.loc["Average"] = average_row

# Export the clean matrix straight to file
pivot_df.to_csv("llm_optimization_results.csv")

# Display results
pivot_df.head(10)

Baseline                                                        \
         Latency_Sec RAM_Used_MB Output_Word_Count  ROUGE1  ROUGE2  ROUGEL   
Paper_ID                                                                     
0              1.235         0.2              53.0  0.3636  0.1494  0.2614   
1              0.875         0.0              45.0  0.4061  0.1846  0.2335   
2              0.965         0.0              49.0  0.3953  0.1529  0.2209   
3              1.354         0.0              93.0  0.4033  0.1222  0.2486   
4              1.355         0.0              73.0  0.5114  0.2120  0.3653   
5              1.406         0.0              95.0  0.3048  0.0673  0.1524   
6              1.294         0.0              81.0  0.1702  0.0870  0.1277   
7              1.137         0.0              67.0  0.3976  0.1585  0.2771   
8              1.206         0.0              67.0  0.4302  0.1749  0.2566   
9              1.123         0.0              65.0  0.3380  0.1000  0.2394   

                   Quantized_Model                                ...  \
         BERTScore     Latency_Sec RAM_Used_MB Output_Word_Count  ...   
Paper_ID                                                          ...   
0              0.0           0.950         0.0              45.0  ...   
1              0.0           1.765         0.0             106.0  ...   
2              0.0           2.025         0.0             106.0  ...   
3              0.0           0.979         0.0              50.0  ...   
4              0.0           2.327         0.0             142.0  ...   
5              0.0           2.599         0.0             172.0  ...   
6              0.0           1.068         0.0              54.0  ...   
7              0.0           1.790         0.0             110.0  ...   
8              0.0           1.962         0.0             127.0  ...   
9              0.0           2.602         0.0             154.0  ...   

         Prompt_Compression                   Quantized_Prompt_Compression  \
                     ROUGE2  ROUGEL BERTScore                  Latency_Sec   
Paper_ID                                                                     
0                    0.0825  0.2048       0.0                        2.494   
1                    0.1132  0.2150       0.0                        1.369   
2                    0.1622  0.2411       0.0                        1.284   
3                    0.0899  0.2179       0.0                        2.002   
4                    0.1565  0.2586       0.0                        1.032   
5                    0.0222  0.1319       0.0                        1.095   
6                    0.1075  0.1474       0.0                        1.338   
7                    0.1757  0.2533       0.0                        2.471   
8                    0.1141  0.1800       0.0                        1.871   
9                    0.0299  0.1324       0.0                        1.407   

                                                                          
         RAM_Used_MB Output_Word_Count  ROUGE1  ROUGE2  ROUGEL BERTScore  
Paper_ID                                                                  
0               0.00             168.0  0.4014  0.1027  0.2177       0.0  
1               0.00              88.0  0.3884  0.1000  0.1901       0.0  
2               0.03              75.0  0.4545  0.1735  0.2424       0.0  
3               0.00             126.0  0.4635  0.1519  0.2821       0.0  
4               0.00              55.0  0.3535  0.1020  0.1919       0.0  
5               0.00              65.0  0.2088  0.0222  0.0989       0.0  
6               0.00              76.0  0.1591  0.0233  0.0909       0.0  
7               0.00             169.0  0.3284  0.0301  0.1418       0.0  
8               0.00             129.0  0.4817  0.1656  0.2622       0.0  
9               0.00              84.0  0.2963  0.0375  0.1358       0.0  

[10 rows x 28 columns]

In [17]:
# 4th Cell: Extract and pivot the actual text summaries cleanly
results_df = pd.DataFrame(results)

summary_df = results_df.pivot(
    index="Paper_ID",
    columns="Pipeline",
    values="Summary"
)

# Optional cleanup: replace any raw error codes with a cleaner display label
for col in summary_df.columns:
    summary_df[col] = summary_df[col].apply(lambda x: "Inference Error" if "Error during inference:" in str(x) else x)

# Display a preview of the actual sentences inside your notebook
print("Actual Text Summaries Preview:")
print(summary_df.head(2))

# Save the text summaries matrix to a separate CSV
summary_df.to_csv("llm_generated_summaries.csv")
print("\nSuccess! Actual text summaries saved cleanly to 'llm_generated_summaries.csv'.")

Actual Text Summaries Preview:
Pipeline                                           Baseline  \
Paper_ID                                                      
0         TnT is an efficient statistical part-of-speech...   
1         This paper explores various constraints for mi...   

Pipeline                                 Prompt_Compression  \
Paper_ID                                                      
0         The TnT part-of-speech tagger employs second o...   
1         This scientific text examines the balance betw...   

Pipeline                                    Quantized_Model  \
Paper_ID                                                      
0         TnT is an efficient statistical POS tagger bas...   
1         Dependency structures are critical for syntact...   

Pipeline                       Quantized_Prompt_Compression  
Paper_ID                                                     
0         The TnT, a statistical part-of-speech tagger b...  
1         Dependency-bas